# 📘 Notebook 11: Streaming

### Course: *From One Table to a Cluster: Working with Data at Scale*
**Instructor:** Ioannis Tsioulis

*Module D: Data in Motion and in the Cloud · Notebook 1 of 2 in this module · (11 of 12 overall)*

---

## 🎯 Learning objectives

By the end of this notebook you will be able to:

- Explain how **unbounded** data differs from the files and tables of the earlier notebooks
- Cut a stream into **tumbling**, **sliding** and **session** windows, by hand
- Distinguish **event time** from **processing time**, and show what goes wrong when they are confused
- Use a **watermark** to decide when a window is finished, and measure the trade between waiting and losing data
- Handle events that arrive **twice**, without unbounded memory
- Run the same windowed computation with Spark Structured Streaming

> **Prerequisites:** Notebook 07 (processing a stream in one pass with a small state, generators). Notebook 08 (pure functions, running work again after a failure). Notebook 09 for the final section.
>
> 🔭 **Why this notebook matters:** every dataset so far had an end. We could read it all, and then answer. A great deal of the data that matters most is not like that: payments, clicks, sensor readings, the positions of delivery vans. It never stops arriving, and the people who depend on it cannot wait until tomorrow's batch to learn that a payment system failed at noon. Computing on data that has no end forces us to face a question that never arose before: **when is an answer finished?** The uncomfortable truth of streaming is that it never quite is, and the craft lies in handling that well.

> 📦 **Libraries used in this notebook.**
>
> - Nearly everything is written by hand in plain Python. **Matplotlib** draws two charts.
> - Section 7 uses **PySpark** (Notebook 09) to run the same computation with **Structured Streaming**, Spark's interface for unbounded data. Its installation line is in the cell that uses it.
>
> The cell below installs what is needed. In Google Colab it is already present and the cell finishes in a moment.

In [ ]:
%pip install -q matplotlib

In [ ]:
import math
import random
from collections import defaultdict

import matplotlib.pyplot as plt

## 1. Data without an end

### Intuition first
A file is a lake: all the water is there, and you can measure its volume whenever you like. A stream is a river. You cannot measure "the volume of the river", because it has not finished flowing and never will. You can only ask questions that have an answer **now**: how much water passed in the last hour, how fast it is flowing at this moment, whether the level has risen since yesterday.

### Formal definition
A dataset is **bounded** if it is finite and complete: a file, a table. It is **unbounded** if it is produced continuously with no defined end. An item of an unbounded dataset is usually called an **event**: a small record of something that happened, carrying the **time** at which it happened.

Processing bounded data, all at once, is **batch processing**: everything in this course up to now. Processing unbounded data, as it arrives, is **stream processing**.

Our stream is the bookshop's orders as they are placed. Each event has an identifier, the time of the order in seconds since the shop opened, a customer, a category and an amount.

In [ ]:
def make_events(count=5000, seed=0):
    rng = random.Random(seed)
    categories = ["Fiction", "Science", "History", "Children", "Cooking", "Travel", "Art", "Computing"]
    events, clock = [], 0.0
    for event_id in range(count):
        clock += rng.expovariate(1 / 1.2)                 # on average, an order every 1.2 seconds
        events.append({"id": event_id, "time": round(clock, 1), "customer": rng.randint(1, 300),
                       "category": rng.choice(categories), "amount": round(rng.uniform(5, 60), 2)})
    return events

events = make_events()
print(f"{len(events):,} events over {events[-1]['time'] / 60:.0f} minutes")
for event in events[:3]:
    print(event)

For the experiments we hold these events in a list, so that we can check our answers against the truth. Every function below is written, though, as if the events were arriving one at a time and the future were unknown: each takes a stream and looks at each event once, as in Notebook 07.

## 2. Windows

### Intuition first
*"How many orders have there been?"* has no final answer on a stream: the number keeps going up. A question that does have an answer is *"how many orders were there **in each minute**?"*. Once a minute is over, its count is settled and can be reported.

Cutting an endless stream into finite pieces of time is called **windowing**, and each piece is a **window**. It is the device that turns an unanswerable question about a stream into a series of answerable ones.

### Tumbling windows
The simplest kind: windows of a fixed length, placed end to end, with no gaps and no overlap. Every event belongs to exactly one.

```
 time  ──────────────────────────────────────────────────►
 events   x  x x    x   x xx  x     x  x x   x    x  x x
 windows [---- 1 ----][---- 2 ----][---- 3 ----][---- 4 --
```

To find the window of an event, divide its time by the length of the window and round down. The function below reports each window as soon as an event arrives that lies beyond it.

In [ ]:
def tumbling_counts(stream, size):
    """Yield (start of the window, number of events) for each window, as soon as it is over."""
    current, count = None, 0
    for event in stream:
        window = math.floor(event["time"] / size) * size
        if current is None:
            current = window
        while window > current:                    # the event lies beyond the current window: report and move on
            yield (current, count)
            current += size
            count = 0
        count += 1
    if current is not None:
        yield (current, count)

per_minute = list(tumbling_counts(iter(events), size=60))
print("First windows (start in seconds, orders):", per_minute[:6])

plt.figure(figsize=(9, 3))
plt.bar([start / 60 for start, count in per_minute], [count for start, count in per_minute], width=0.9)
plt.title("Orders in each minute (tumbling windows of 60 seconds)")
plt.xlabel("Minutes since the shop opened")
plt.ylabel("Orders")
plt.grid(True, axis="y")
plt.show()

The state kept by `tumbling_counts` is one window and one number, whatever the length of the stream. Its memory is constant, which for a stream that never ends is not an optimisation but a necessity.

### Sliding windows
A count per minute jumps about a good deal. A manager watching a screen would prefer *"orders in the last five minutes"*, refreshed every minute. Those windows **overlap**: each is five minutes long, and a new one starts every minute.

```
 window 1 [---------------]
 window 2      [---------------]
 window 3           [---------------]
```

An event now belongs to several windows: every window that started at or before it and has not yet ended.

In [ ]:
def sliding_counts(stream, size, slide):
    """Count events in windows of length `size` that start every `slide` seconds."""
    counts = defaultdict(int)
    for event in stream:
        last_start = math.floor(event["time"] / slide) * slide          # the latest window that contains the event
        start = last_start
        while start > event["time"] - size:                             # and every earlier one that still contains it
            if start >= 0:
                counts[start] += 1
            start -= slide
    return dict(sorted(counts.items()))

smooth = sliding_counts(iter(events), size=300, slide=60)

plt.figure(figsize=(9, 3))
plt.plot([start / 60 for start, count in per_minute], [count * 5 for start, count in per_minute], alpha=0.5, label="per minute, multiplied by 5")
plt.plot([(start + 300) / 60 for start in smooth], list(smooth.values()), linewidth=2, label="last 5 minutes, updated every minute")
plt.title("Tumbling against sliding windows")
plt.xlabel("Minutes since the shop opened")
plt.ylabel("Orders per 5 minutes")
plt.legend()
plt.grid(True)
plt.show()

The sliding count is the same information with the jitter averaged away. (This simple version keeps all its counts. A production one would report and forget each window once it had ended, as `tumbling_counts` does.)

### Session windows
The third kind is not defined by the clock at all. A **session** is a burst of activity by one customer: a run of events with no long gap between them. A session window stays open as long as events keep coming, and closes after a chosen period of silence.

In [ ]:
def sessions(stream, gap):
    """Group each customer's events into sessions separated by more than `gap` seconds of silence."""
    open_sessions = {}                    # customer -> [time of first event, time of last event, number of events]
    finished = []
    for event in stream:
        customer, now = event["customer"], event["time"]
        if customer in open_sessions and now - open_sessions[customer][1] > gap:
            finished.append((customer, *open_sessions.pop(customer)))   # silence was too long: the session is over
        if customer not in open_sessions:
            open_sessions[customer] = [now, now, 0]
        open_sessions[customer][1] = now
        open_sessions[customer][2] += 1
    for customer, session in open_sessions.items():
        finished.append((customer, *session))
    return finished

found = sessions(iter(events), gap=120)
longest = max(found, key=lambda session: session[3])
print(f"{len(found):,} sessions from {len(set(session[0] for session in found))} customers")
print(f"Largest: customer {longest[0]} placed {longest[3]} orders between second {longest[1]:.0f} and second {longest[2]:.0f}")

| Window | Defined by | An event belongs to | Typical question |
|---|---|---|---|
| **Tumbling** | a fixed length | exactly one window | orders in each hour |
| **Sliding** | a length and a step | several overlapping windows | orders in the last hour, updated every minute |
| **Session** | a gap of inactivity | one session of its own key | how long does a visit to the site last? |

## 3. Two kinds of time

### Intuition first
Everything so far assumed that events reach us in the order in which they happened, the instant they happened. They do not.

A customer orders on a phone in a train tunnel, and the order is sent three minutes later when the signal returns. A shop's till loses its connection and uploads an hour of sales at once. Two servers have clocks that disagree by a second. In any real system there are two different times attached to every event.

- **Event time**: when the thing actually happened. It is stamped on the event at its source.
- **Processing time**: when our system gets to see it.

The difference between the two is the **delay**. It varies from event to event, and it is occasionally very large.

Let us give our orders realistic delays: most arrive within a couple of seconds, some within a minute, and a few much later.

In [ ]:
def with_delays(events, seed=1):
    rng = random.Random(seed)
    arriving = []
    for event in events:
        roll = rng.random()
        if roll < 0.90:
            delay = rng.expovariate(1 / 2)             # most: a second or two
        elif roll < 0.98:
            delay = rng.uniform(10, 60)                # some: up to a minute
        else:
            delay = rng.uniform(60, 300)               # a few: several minutes
        arriving.append(dict(event, arrival=round(event["time"] + delay, 1)))
    arriving.sort(key=lambda event: event["arrival"])  # the order in which we actually see them
    return arriving

arriving = with_delays(events)

out_of_order = sum(1 for earlier, later in zip(arriving, arriving[1:]) if later["time"] < earlier["time"])
print(f"{out_of_order:,} of {len(arriving):,} events arrive after an event that happened later than they did")
print("\nA stretch of the stream, in order of arrival:")
for event in arriving[200:206]:
    print(f"  arrived at {event['arrival']:>6.1f}   happened at {event['time']:>6.1f}   delay {event['arrival'] - event['time']:>5.1f} s")

The stream we receive is **out of order**. What happens if we ignore that, and simply count each event in the minute in which it **arrives**?

In [ ]:
def counts_by(events, field, size=60):
    counts = defaultdict(int)
    for event in events:
        counts[math.floor(event[field] / size) * size] += 1
    return counts

truth = counts_by(arriving, "time")                    # by event time: what really happened in each minute
by_arrival = counts_by(arriving, "arrival")            # by processing time: what we would report

wrong = sum(1 for window in truth if truth[window] != by_arrival.get(window, 0))
misplaced = sum(abs(truth[window] - by_arrival.get(window, 0)) for window in truth) / 2

print(f"Minutes with a wrong count: {wrong} of {len(truth)}")
print(f"Orders counted in the wrong minute: about {misplaced:.0f} of {len(arriving):,}")

Most minutes are reported wrongly. The total is right, since every event is counted somewhere, but orders have been credited to the minute in which the network delivered them and not the minute in which the customer placed them.

For a rough picture of traffic this may not matter. For anything that must be correct **about the world**, such as billing by the hour, detecting a failure at a particular moment, or comparing this hour with the same hour last week, it is simply the wrong answer. Results must be computed by **event time**.

That immediately raises the difficulty of this whole subject. If events can arrive late, then when the clock passes the end of a window, we cannot know that we have seen all of its events. So **when may we report a window as finished?**

## 4. Watermarks: deciding when to stop waiting

### Intuition first
A teacher collecting homework faces the same problem. The deadline is nine o'clock. At nine, some pupils are still in the corridor. If she starts marking at nine sharp, she marks at once, and the latecomers get nothing. If she waits until every pupil in the school has certainly handed in, she never starts. So she picks a rule: *"I wait ten minutes, and after that the box is closed."*

A stream processor makes the same decision, and the name for its rule is a **watermark**.

### Formal definition
A **watermark** is the system's running estimate of how far event time has progressed: a claim that *"no more events with a time earlier than this are expected"*. A common, simple watermark is

$$\text{watermark} = (\text{latest event time seen so far}) - (\text{allowed lateness})$$

When the watermark passes the end of a window, the window is **closed** and its result is reported. An event that arrives for a window already closed is **late**, and by default it is dropped.

In [ ]:
def windowed_counts(arriving, size=60, allowed_lateness=0.0):
    """Count events per window of event time, closing each window when the watermark passes its end."""
    open_windows = defaultdict(int)
    results, waited = {}, []
    dropped, latest_time = 0, 0.0

    for event in arriving:                                  # events come in order of ARRIVAL
        latest_time = max(latest_time, event["time"])
        watermark = latest_time - allowed_lateness
        window = math.floor(event["time"] / size) * size

        if window + size <= watermark:
            dropped += 1                                    # its window was already closed: too late
        else:
            open_windows[window] += 1

        for start in [start for start in open_windows if start + size <= watermark]:
            results[start] = open_windows.pop(start)        # the watermark has passed: report this window
            waited.append(event["arrival"] - (start + size))    # how long after the window ended did we report it?

    results.update(open_windows)                            # the stream of our experiment ends here: report what is left
    return results, dropped, sum(waited) / len(waited)

results, dropped, wait = windowed_counts(arriving, allowed_lateness=0)
exact = sum(1 for window in truth if results.get(window) == truth[window])

print(f"With no allowed lateness: {dropped} events dropped, {exact} of {len(truth)} minutes exactly right,")
print(f"each window reported on average {wait:.0f} seconds after it ended")

With no patience at all, results come out almost at once, and a noticeable share of the events is thrown away because their window had closed before they arrived. Now let us vary the patience.

In [ ]:
print(f"{'allowed lateness':>17} {'events dropped':>15} {'minutes exactly right':>22} {'wait for a result':>18}")
latenesses = [0, 5, 15, 30, 60, 120, 300]
dropped_share, waits = [], []
for lateness in latenesses:
    results, dropped, wait = windowed_counts(arriving, allowed_lateness=lateness)
    exact = sum(1 for window in truth if results.get(window) == truth[window])
    dropped_share.append(100 * dropped / len(arriving))
    waits.append(wait)
    print(f"{lateness:>15} s {dropped:>9} ({dropped / len(arriving):>4.1%}) {exact:>15} of {len(truth)} {wait:>16.0f} s")

In [ ]:
figure, left = plt.subplots(figsize=(7, 4))
left.plot(latenesses, dropped_share, marker="o", color="tab:red")
left.set_xlabel("Allowed lateness (seconds)")
left.set_ylabel("Events dropped (%)", color="tab:red")
right = left.twinx()
right.plot(latenesses, waits, marker="s", color="tab:blue")
right.set_ylabel("Average wait for a result (seconds)", color="tab:blue")
plt.title("The trade at the heart of streaming")
left.grid(True)
plt.show()

The two lines move in opposite directions, and nothing can make both low at once.

| | A short wait | A long wait |
|---|---|---|
| Results arrive | almost immediately | after a delay |
| Late events | many are dropped | few are dropped |
| The counts are | fresh and incomplete | complete and stale |
| Suits | a live dashboard, an alarm | billing, official figures |

This is the answer to the question of when a window is finished: **it is finished when we decide to stop waiting**, and whatever we decide, we pay in one currency or the other. No setting is right in general. An alarm that fires five minutes late is useless, and a few missing events do not matter to it. An invoice that omits 4% of a customer's orders is unacceptable, and nobody minds if it is produced an hour later.

Real systems soften the choice in two ways. A window can be reported early and then **corrected** when late events turn up, so that the first figure is quick and the last is right. And events too late even for that can be sent to a separate place and not silently dropped, so that they can be counted in a nightly batch. The nightly batch, working on data that has had a day to settle, is still how most organisations produce the figures that must be exactly right.

> ⚠️ **Common pitfalls**
>
> - Windowing by the time of arrival because it is easier. The results look plausible and describe the network, not the world.
> - Choosing an allowed lateness by guesswork. Measure the real distribution of delays in your own data, as we did, and choose with the trade in front of you.
> - Assuming that the clocks of different machines agree. They do not, quite. Event times set by many sources are only approximately comparable.
> - Forgetting the quiet stream. A watermark that advances only when events arrive will never close the last window if the events stop. Real systems also advance it with the passing of time.

## 5. Events that arrive twice

A stream processor is a program on a machine, and machines fail. In Notebook 08 the remedy for a failure was to **run the work again**. In a stream that has a consequence. Suppose a sender transmits an order, the connection drops before the acknowledgement comes back, and the sender, not knowing whether the order arrived, sends it again. Now the order may be in the stream twice.

A system can offer one of three guarantees about each event.

- **At most once**: never repeat anything. Simple, and events are lost when something fails.
- **At least once**: repeat until sure. Nothing is lost, and some events are processed twice.
- **Exactly once**: every event affects the result precisely once. This is what everybody wants, and it has to be built.

Let us produce an at-least-once stream, in which 3% of the events are sent a second time a little later, and add up the revenue.

In [ ]:
def with_duplicates(arriving, share=0.03, seed=2):
    rng = random.Random(seed)
    stream = list(arriving)
    for event in arriving:
        if rng.random() < share:
            stream.append(dict(event, arrival=round(event["arrival"] + rng.uniform(1, 30), 1)))     # sent again
    stream.sort(key=lambda event: event["arrival"])
    return stream

at_least_once = with_duplicates(arriving)
true_revenue = sum(event["amount"] for event in events)
naive_revenue = sum(event["amount"] for event in at_least_once)

print(f"Events in the stream: {len(at_least_once):,}   (true number of orders: {len(events):,})")
print(f"True revenue:    {true_revenue:>12,.2f}")
print(f"Counted naively: {naive_revenue:>12,.2f}   (too high by {naive_revenue - true_revenue:,.2f})")

The shop appears to have earned money it did not earn. The cure is the one from Notebook 10: make the processing **idempotent**, so that handling the same event twice has the same effect as handling it once. Each event carries a unique identifier. Remember the identifiers already seen, and ignore any that comes again.

In [ ]:
def revenue_exactly_once(stream):
    seen = set()
    total = 0.0
    for event in stream:
        if event["id"] in seen:
            continue                           # a repeat: ignore it
        seen.add(event["id"])
        total += event["amount"]
    return total, len(seen)

total, remembered = revenue_exactly_once(at_least_once)
print(f"With deduplication: {total:>12,.2f}   correct: {abs(total - true_revenue) < 0.01}")
print(f"Identifiers remembered: {remembered:,}")

Correct. But look at the last line, and think about a stream that never ends. The set of identifiers grows for ever. After a year it holds every order ever placed, and we have rebuilt, inside a "constant memory" stream processor, the very thing Notebook 07 taught us to avoid.

The way out uses the idea of the previous section. A duplicate is sent soon after the original. If we are willing to say *"a repeat will not come more than ten minutes late"*, then identifiers older than that can safely be forgotten.

In [ ]:
def revenue_with_bounded_memory(stream, remember_for=600):
    seen = {}                                  # identifier -> the time we saw it
    total, largest = 0.0, 0
    for event in stream:
        now = event["arrival"]
        if event["id"] not in seen:
            total += event["amount"]
        seen[event["id"]] = now
        if len(seen) > 2000:                   # from time to time, forget what is too old to be repeated
            seen = {identifier: time for identifier, time in seen.items() if now - time <= remember_for}
        largest = max(largest, len(seen))
    return total, largest

total, largest = revenue_with_bounded_memory(at_least_once)
print(f"Revenue: {total:>12,.2f}   correct: {abs(total - true_revenue) < 0.01}")
print(f"Most identifiers held at any moment: {largest:,}")

The same correct total, with a memory that stays bounded however long the stream runs. Once more the price is an assumption about lateness: a duplicate arriving after the limit would be counted twice. In streaming, **every guarantee comes with a time limit attached**.

## 6. Batch or stream?

| | Batch processing | Stream processing |
|---|---|---|
| Data | bounded: a complete file or table | unbounded: events as they arrive |
| An answer is | computed once, on everything, and final | updated continuously, and provisional |
| How fresh | as old as the last run: hours | seconds |
| How complete | complete, since late data has had time to arrive | depends on how long you wait |
| Complexity | low: read, compute, write | high: time, lateness, state, duplicates |
| When something fails | run the job again | recover the state and continue, without losing or repeating events |
| Suits | reports, billing, training models, anything that can wait | alarms, live dashboards, fraud detection, anything that cannot |

This notebook has shown why the right-hand column is so much harder. A batch job has none of the problems of Sections 3 to 5: by the time it runs, everything has arrived, and if it fails it is simply run again.

The practical advice follows. **Use streaming only for what must be fresh.** Many "real-time" requirements turn out, when someone asks, to mean "within the hour", and a batch job every ten minutes meets them at a fraction of the cost and difficulty. Where both freshness and exactness matter, organisations commonly run both: a stream for quick provisional figures, and a batch job that later produces the definitive ones.

## 7. The same windows in Spark

*This section runs Spark Structured Streaming and writes files, and is meant for Colab.*

Spark handles unbounded data with the DataFrame interface of Notebook 09. A stream is treated as a **table that keeps growing**, and a query on it is written exactly like a batch query. Spark keeps the query running and updates the result as new rows arrive.

To imitate a live source, we write our arriving events as twenty small files and tell Spark to take one file at a time, as though they were being delivered one after another.

In [ ]:
%pip install -q pyspark
import csv
import os
import tempfile

from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = (SparkSession.builder.master("local[*]").appName("streaming")
         .config("spark.sql.shuffle.partitions", "2")
         .config("spark.ui.showConsoleProgress", "false").getOrCreate())
spark.sparkContext.setLogLevel("ERROR")

stream_folder = tempfile.mkdtemp()
per_file = len(arriving) // 20
for number in range(20):
    with open(os.path.join(stream_folder, f"events_{number:02d}.csv"), "w", newline="") as file:
        writer = csv.writer(file)
        writer.writerow(["id", "time", "customer", "category", "amount"])
        for event in arriving[number * per_file:(number + 1) * per_file]:
            writer.writerow([event["id"], event["time"], event["customer"], event["category"], event["amount"]])
print("20 files written to", stream_folder)

In [ ]:
incoming = (spark.readStream                                    # readStream, where a batch job says read
            .schema("id INT, time DOUBLE, customer INT, category STRING, amount DOUBLE")
            .option("header", True)
            .option("maxFilesPerTrigger", 1)                    # take one file at a time
            .csv(stream_folder)
            .withColumn("event_time", F.col("time").cast("timestamp")))

per_window = (incoming.groupBy(F.window("event_time", "5 minutes"))        # tumbling windows of event time
                      .agg(F.count("*").alias("orders"), F.round(F.sum("amount"), 2).alias("revenue")))

query = (per_window.writeStream
                   .outputMode("complete")                      # after each file, publish the whole updated table
                   .format("memory")
                   .queryName("orders_per_window")
                   .start())

query.processAllAvailable()                                     # wait until every file has been taken in
spark.sql("SELECT window.start, window.end, orders, revenue FROM orders_per_window ORDER BY start").show(6, truncate=False)
query.stop()

In [ ]:
by_hand = defaultdict(int)
for event in arriving:
    by_hand[math.floor(event["time"] / 300) * 300] += 1
print("Our own count for the first windows of 5 minutes:", [by_hand[start] for start in sorted(by_hand)[:6]])
spark.stop()

The counts agree with those of a window function written by hand in a dozen lines. `F.window("event_time", "5 minutes")` is `math.floor(time / size) * size`. The allowed lateness of Section 4 is one more method in the chain, `.withWatermark("event_time", "1 minute")`, and sliding windows are a third argument to `F.window`.

What Spark adds to our dozen lines is everything this notebook did not build: the state of every open window kept safely across machines and across failures, events from many sources merged, and each event counted exactly once. The concepts, though, are the ones you have now implemented yourself.

## 8. Where we stand

| The problem | The answer |
|---|---|
| The data never ends, so there is never an "all of it" | windows: finite pieces of time |
| Events arrive late and out of order | compute by event time, not by arrival |
| We cannot know when a window is complete | a watermark: decide how long to wait |
| Waiting longer loses less and delays more | choose by what the result is for |
| After a failure, events may come twice | idempotent processing, with a bounded memory of what was seen |

Each earlier module asked what to do when the data no longer fitted in a table, in time, in memory, on a machine. Here it no longer fitted in **time itself**: there is no moment at which the data is all present. The response was the same as ever. Give up something we had taken for granted, in this case the idea of a final answer, and gain the ability to keep going.

One notebook remains. It puts the pieces of the course together into a single working pipeline, and asks where, in practice, all of this runs, and what it costs.

---
## ✏️ Exercises

### Exercise 1 (Revenue per window)
Write `tumbling_revenue(stream, size)` that returns, for each tumbling window, the total `amount` of its events. Run it on `iter(events)` with windows of ten minutes and print the three windows with the highest revenue.

In [ ]:
# Your solution here:

<details><summary>💡 Show solution</summary>

```python
def tumbling_revenue(stream, size):
    totals = defaultdict(float)
    for event in stream:
        totals[math.floor(event["time"] / size) * size] += event["amount"]
    return totals

revenue = tumbling_revenue(iter(events), 600)
for start, total in sorted(revenue.items(), key=lambda item: -item[1])[:3]:
    print(f"minutes {start / 60:>3.0f} to {start / 60 + 10:>3.0f}: {total:>10,.2f}")
```

*The window of an event is found by the same division whatever is being aggregated. Only the last line of the loop changes.*
</details>

### Exercise 2 (A window per category)
Count the orders per **category** in each tumbling window of 15 minutes. The key of each count is now a pair: the window and the category. Print the counts of the first window.

In [ ]:
# Your solution here:

<details><summary>💡 Show solution</summary>

```python
counts = defaultdict(int)
for event in events:
    window = math.floor(event["time"] / 900) * 900
    counts[(window, event["category"])] += 1

for (window, category), count in sorted(counts.items())[:8]:
    print(f"window starting at {window:>5} s   {category:<10} {count}")
```

*This is `GROUP BY window, category`. A window is one more column to group by, which is exactly how Spark treats it in Section 7.*
</details>

### Exercise 3 (Sessions)
Using `sessions`, compare a gap of 30 seconds, 120 seconds and 600 seconds. For each, print the number of sessions and the average number of orders in a session. What happens as the gap grows, and what does that say about choosing it?

In [ ]:
# Your solution here:

<details><summary>💡 Show solution</summary>

```python
print(f"{'gap':>6} {'sessions':>9} {'orders per session':>19}")
for gap in [30, 120, 600]:
    found = sessions(iter(events), gap)
    print(f"{gap:>4} s {len(found):>9,} {sum(session[3] for session in found) / len(found):>19.2f}")
```

*A short gap cuts activity into many small sessions, and a long one merges separate visits into a few large ones. The number of "visits" reported is therefore partly a product of the setting. There is no natural gap in the data: it is a definition, and it should be chosen with the people who will read the figures.*
</details>

### Exercise 4 (Choose a watermark)
A dashboard must show each minute's orders within 45 seconds of the minute ending, on average. Billing may wait up to ten minutes, and must lose fewer than 0.5% of the events. Using `windowed_counts` on `arriving`, find a suitable allowed lateness for each of the two uses, by trying the values 0, 10, 20, 30, 45, 60, 120, 240 and 400 seconds.

In [ ]:
# Your solution here:

<details><summary>💡 Show solution</summary>

```python
print(f"{'lateness':>9} {'dropped':>9} {'average wait':>13}")
for lateness in [0, 10, 20, 30, 45, 60, 120, 240, 400]:
    results, dropped, wait = windowed_counts(arriving, allowed_lateness=lateness)
    notes = []
    if wait <= 45:
        notes.append("fast enough for the dashboard")
    if dropped / len(arriving) < 0.005 and wait <= 600:
        notes.append("complete enough for billing")
    print(f"{lateness:>7} s {dropped / len(arriving):>8.2%} {wait:>11.0f} s   {', '.join(notes)}")
```

*No single setting satisfies both. The dashboard needs a short lateness and accepts some loss. Billing needs a long one and accepts the wait. The usual solution is to run the same computation twice with different settings, which costs little, since the logic is identical.*
</details>

### Exercise 5 (How long to remember? A little harder)
In `with_duplicates` a repeat arrives between 1 and 30 seconds after the original. Run `revenue_with_bounded_memory` on `at_least_once` with `remember_for` set to 5, 15, 30, 60 and 600 seconds, and print the error in the total and the largest number of identifiers held. Note that the function only clears its memory when it holds more than 2,000 identifiers. Change that limit to 200 for this experiment (copy the function and modify it). What is the smallest safe value, and why?

In [ ]:
# Your solution here:

<details><summary>💡 Show solution</summary>

```python
def revenue_remembering(stream, remember_for, clear_above=200):
    seen, total, largest = {}, 0.0, 0
    for event in stream:
        now = event["arrival"]
        if event["id"] not in seen:
            total += event["amount"]
        seen[event["id"]] = now
        if len(seen) > clear_above:
            seen = {identifier: time for identifier, time in seen.items() if now - time <= remember_for}
        largest = max(largest, len(seen))
    return total, largest

print(f"{'remember for':>13} {'error in revenue':>17} {'most identifiers held':>22}")
for remember_for in [5, 15, 30, 60, 600]:
    total, largest = revenue_remembering(at_least_once, remember_for)
    print(f"{remember_for:>11} s {total - true_revenue:>17,.2f} {largest:>22,}")
```

*With a memory shorter than the longest possible gap between an original and its repeat, some repeats are no longer recognised and are counted again. Thirty seconds, the longest gap in this stream, is the smallest safe value, and remembering longer only costs memory. In a real system the gap is not known exactly. One measures it, adds a generous margin, and accepts that a repeat beyond the margin will slip through.*
</details>

## 🔑 Key takeaways

- **Unbounded** data has no end, so a computation can never wait for "all of it". **Windows** cut it into finite pieces.
- **Tumbling** windows are fixed and adjacent, **sliding** windows overlap, and **session** windows follow bursts of activity.
- **Event time** is when something happened. **Processing time** is when we see it. Correct results about the world need event time.
- A **watermark** decides when a window is closed. Waiting longer drops fewer late events and delays every result. There is no setting that avoids the trade.
- After failures, events can arrive twice. **Idempotent** processing with remembered identifiers gives the effect of exactly once, and the memory must be bounded by an assumption about lateness.
- Streaming is much harder than batch. Use it for what must be **fresh**, and batch for what must be **exact**.

---
**Next:** *Notebook 12: The Cloud and an End-to-End Pipeline*, the final notebook, where everything in the course is assembled into one working system, and we count what it costs to run.

---
*© Ioannis Tsioulis. *From One Table to a Cluster: Working with Data at Scale*. Prepared for educational use.*